# Multimodal Evaluation v2 — Index Reconstruction

Cache'te dosya isimleri yok ama W (waveforms), y (labels), splits var. Bu notebook:

1. RAVDESS dosya isimlerini deterministic üretir (1056 dosya)
2. CREMA-D dosya isimlerini GitHub API'den çeker (~6171 dosya)
3. Sıralayıp **cache.y ile karşılaştırarak doğrular** — etiketler eşleşmeli
4. Test split'i belirler
5. SER eval (cache direkt, ~%69 sanity check)
6. Test video'larını indirir
7. Vision eval
8. Fusion + final metrikler

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob, urllib.request
import numpy as np
import pandas as pd
import tensorflow as tf
import librosa
import cv2
from tqdm import tqdm
from collections import Counter

gpus = tf.config.list_physical_devices('GPU')
print(f'TF {tf.__version__}, GPU: {gpus if gpus else "YOK (CPU)"}')

In [ ]:
# Paths
DRIVE      = '/content/drive/MyDrive/BitirmeProjesi'
SER_DIR    = f'{DRIVE}/ser_cnn_bilstm'
VISION_DIR = f'{DRIVE}/vision_ferplus'  # vision dosyan başka yerdeyse güncelle

SER_MODEL    = f'{SER_DIR}/ser_cnn_bilstm.keras'
SER_META     = f'{SER_DIR}/ser_cnn_bilstm_meta.json'
SER_CACHE    = f'{SER_DIR}/waveforms_cache.npz'
VISION_MODEL = f'{VISION_DIR}/vision_ferplus.keras'

WORK = '/content/eval_work'
RAV_VIDEO_DIR = f'{WORK}/ravdess_video'
CRE_VIDEO_DIR = f'{WORK}/crema_video'
OUT_DIR = f'{DRIVE}/multimodal_eval'
for d in [WORK, RAV_VIDEO_DIR, CRE_VIDEO_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)

CLASSES = ['Angry', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
NUM_CLASSES = 6
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# Fusion weights (live = 0.50/0.35/0.20, NLP=0, renormalize)
W_VISION = 0.50 / 0.85
W_SER    = 0.35 / 0.85
print(f'Fusion: vision={W_VISION:.3f}, ser={W_SER:.3f}')

# Audio params (training ile aynı)
SR, N_MELS, N_FFT, HOP = 16000, 128, 1024, 256
TARGET_LEN = 48000
VISION_IMG_SIZE = 48
FRAMES_PER_CLIP = 8

In [ ]:
# Modelleri ve cache'i yükle
vision_model = tf.keras.models.load_model(VISION_MODEL)
ser_model    = tf.keras.models.load_model(SER_MODEL)
print(f'Vision: {vision_model.count_params():,} params')
print(f'SER:    {ser_model.count_params():,} params')

cache = np.load(SER_CACHE, allow_pickle=True)
W      = cache['W']         # (7227, 48000)
y      = cache['y']         # (7227,)
splits = cache['splits']    # (7227,) -> 'train'/'val'/'test'

print(f'\nCache: W={W.shape}, y={y.shape}, splits={splits.shape}')
split_counts = Counter(splits.tolist())
print(f'Split dağılımı: {dict(split_counts)}')

test_mask_cache = (splits == 'test')
test_indices_cache = np.where(test_mask_cache)[0]
print(f'Test örneği sayısı: {len(test_indices_cache)}')

# Face cascade
HAAR = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(HAAR)

## 2. Dosya isimlerini üret

**RAVDESS**: 24 actor, deterministic naming
**CREMA-D**: GitHub Tree API'den

In [ ]:
# RAVDESS filename pattern: 03-01-{emo}-{int}-{stmt}-{rep}-{actor}.wav
# emo: 01=neutral, 02=calm (drop), 03=happy, 04=sad, 05=angry,
#      06=fearful, 07=disgust (drop), 08=surprised
RAV_EMO_TO_CLASS = {
    '01': 'Neutral', '03': 'Happy', '04': 'Sad',
    '05': 'Angry',   '06': 'Fear',  '08': 'Surprise',
}

rav_files = []
for actor in range(1, 25):
    for emo in sorted(RAV_EMO_TO_CLASS.keys()):
        intensities = ['01'] if emo == '01' else ['01', '02']  # neutral: only normal
        for intensity in intensities:
            for stmt in ['01', '02']:
                for rep in ['01', '02']:
                    fname = f'03-01-{emo}-{intensity}-{stmt}-{rep}-{actor:02d}.wav'
                    label = CLASS_TO_IDX[RAV_EMO_TO_CLASS[emo]]
                    rav_files.append({'filename': fname, 'label': label,
                                       'source': 'ravdess', 'actor': actor})

rav_df = pd.DataFrame(rav_files)
print(f'RAVDESS: {len(rav_df)} dosya')
print(rav_df['label'].value_counts().sort_index().to_dict())
print(rav_df.head(3))

In [ ]:
# CREMA-D: GitHub Tree API ile tek call'da tüm dosya listesi
import requests

CRE_EMO_TO_CLASS = {
    'ANG': 'Angry', 'FEA': 'Fear', 'HAP': 'Happy',
    'NEU': 'Neutral', 'SAD': 'Sad',  # DIS dropped
}

url = 'https://api.github.com/repos/CheyneyComputerScience/CREMA-D/git/trees/master?recursive=1'
print(f'Fetching CREMA-D tree from GitHub...')
r = requests.get(url, timeout=30)
if r.status_code != 200:
    print(f'HATA: {r.status_code}, msg: {r.text[:200]}')
    print('Bir saat bekleyip tekrar dene (rate limit), veya bana traceback at')
    raise SystemExit('GitHub API hatası')

tree = r.json()
cre_audio_files = []
for item in tree['tree']:
    p = item['path']
    if p.startswith('AudioWAV/') and p.endswith('.wav'):
        fname = p.split('/')[-1]
        parts = fname.replace('.wav','').split('_')
        if len(parts) != 4:
            continue
        actor, sent, emo, intensity = parts
        if emo not in CRE_EMO_TO_CLASS:  # skip DIS
            continue
        label = CLASS_TO_IDX[CRE_EMO_TO_CLASS[emo]]
        cre_audio_files.append({'filename': fname, 'label': label,
                                 'source': 'crema_d', 'actor': int(actor)})

cre_df = pd.DataFrame(cre_audio_files)
print(f'CREMA-D: {len(cre_df)} dosya (filtered)')
print(cre_df['label'].value_counts().sort_index().to_dict())
print(cre_df.head(3))

## 3. Sıralamayı bul — cache.y ile eşleşmeli

3 farklı sıralama deneyeceğiz. Hangisi tutarsa devam.

In [ ]:
print(f'Hedef toplam: {len(W)} (cache)')
print(f'Bizim üretim: {len(rav_df) + len(cre_df)}')

if len(rav_df) + len(cre_df) != len(W):
    print(f'!!! UYARI: dosya sayıları uyuşmuyor. RAV={len(rav_df)}, CRE={len(cre_df)}')
    print('!!! CREMA-D listesi eksik olabilir veya emotion mapping yanlış.')

def try_ordering(combined_df, name):
    derived_y = combined_df['label'].values
    if len(derived_y) != len(y):
        return False, 0
    match_pct = (derived_y == y).mean()
    print(f'  {name}: match={match_pct:.4f} ({(derived_y == y).sum()}/{len(y)})')
    return match_pct == 1.0, match_pct

print('\nFarklı sıralamalar deneniyor:')

# A: RAV (sorted) + CRE (sorted)
cand_a = pd.concat([rav_df.sort_values('filename'), cre_df.sort_values('filename')]).reset_index(drop=True)
ok_a, _ = try_ordering(cand_a, 'A: RAV-sorted + CRE-sorted')

# B: CRE (sorted) + RAV (sorted)
cand_b = pd.concat([cre_df.sort_values('filename'), rav_df.sort_values('filename')]).reset_index(drop=True)
ok_b, _ = try_ordering(cand_b, 'B: CRE-sorted + RAV-sorted')

# C: birlikte sıralı
cand_c = pd.concat([rav_df, cre_df]).sort_values('filename').reset_index(drop=True)
ok_c, _ = try_ordering(cand_c, 'C: birlikte filename-sorted')

verified_df = None
if ok_a: verified_df = cand_a; print('\n✓ A doğrulandı')
elif ok_b: verified_df = cand_b; print('\n✓ B doğrulandı')
elif ok_c: verified_df = cand_c; print('\n✓ C doğrulandı')
else:
    print('\n!!! Hiçbir sıralama eşleşmedi. Match yüzdesi >0.9 olanı kullanmayı düşünebiliriz.')
    print('!!! Bu durumda traceback ve match yüzdelerini bana yapıştır.')
    # Devam etmek için en iyi adayı seç (deneysel)
    # raise SystemExit('Sıralama doğrulanamadı')

In [ ]:
# Sıralama doğrulanamadıysa burayı atlamak yerine en iyi adayı kullanmayı dene
if verified_df is None:
    print('Sıralama eşleşmiyor, devam edemiyoruz. Yukarıdaki yüzdeleri bana at.')
    raise SystemExit('Stop')

# Test indeksleri
test_df = verified_df[test_mask_cache].copy().reset_index(drop=True)
test_df['cache_idx'] = test_indices_cache

print(f'Test set: {len(test_df)} klip')
print(f'\nClass dağılımı:')
print(test_df.groupby(['source', test_df['label'].map({i:c for c,i in CLASS_TO_IDX.items()})]).size().unstack(fill_value=0))

## 4. SER eval (cache'ten) — sanity check

Bu adımın çıktısı meta.json'daki **test_accuracy ≈ 0.694** ile birebir eşleşmeli. Eşleşmezse preprocessing'de fark var demektir.

In [ ]:
# Cache W'den log-mel hesapla ve modeli çalıştır
def waveform_to_logmel(y_wav):
    mel = librosa.feature.melspectrogram(y=y_wav, sr=SR, n_fft=N_FFT,
                                          hop_length=HOP, n_mels=N_MELS, power=2.0)
    lm = librosa.power_to_db(mel, ref=np.max)
    lm = (lm - lm.mean()) / (lm.std() + 1e-6)
    return lm.astype(np.float32)

W_test = W[test_indices_cache]
y_test_cache = y[test_indices_cache]
print(f'Computing log-mel for {len(W_test)} test waveforms...')

X_test_mel = np.stack([waveform_to_logmel(w) for w in tqdm(W_test)])[..., None]
print(f'X_test_mel shape: {X_test_mel.shape}')

ser_probs_test = ser_model.predict(X_test_mel, batch_size=32, verbose=1)
ser_pred_test = ser_probs_test.argmax(axis=1)
ser_acc = (ser_pred_test == y_test_cache).mean()
print(f'\nSER test accuracy: {ser_acc:.4f}')
print(f'meta.json test_accuracy: 0.6938')
if abs(ser_acc - 0.6938) > 0.01:
    print('!!! Sanity check FAILED — bu fark dikkat çekici. Preprocessing fark var olabilir.')
else:
    print('✓ Sanity check OK')

## 5. Test video'larını indir

**RAVDESS**: sadece test aktörleri için video zip (Zenodo, ~700MB/aktör)
**CREMA-D**: test dosyaları için tek tek .flv (GitHub raw)

In [ ]:
# RAVDESS video — test aktörler
rav_test = test_df[test_df['source'] == 'ravdess']
rav_test_actors = sorted(rav_test['actor'].unique())
print(f'RAVDESS test aktörleri: {rav_test_actors}')

for actor in rav_test_actors:
    target_dir = f'{RAV_VIDEO_DIR}/Actor_{actor:02d}'
    if os.path.isdir(target_dir) and len(os.listdir(target_dir)) > 0:
        print(f'  Actor {actor:02d}: zaten var, atlandı')
        continue
    zname = f'Video_Speech_Actor_{actor:02d}.zip'
    url = f'https://zenodo.org/record/1188976/files/{zname}'
    local = f'{RAV_VIDEO_DIR}/{zname}'
    print(f'  Actor {actor:02d}: indiriliyor...')
    try:
        urllib.request.urlretrieve(url, local)
        os.system(f'cd "{RAV_VIDEO_DIR}" && unzip -q -o "{zname}" && rm "{zname}"')
        print(f'    OK')
    except Exception as e:
        print(f'    HATA: {e}')

rav_mp4s = glob.glob(f'{RAV_VIDEO_DIR}/**/*.mp4', recursive=True)
print(f'\nToplam RAVDESS video: {len(rav_mp4s)}')

In [ ]:
# CREMA-D video — tek tek .flv indir
cre_test = test_df[test_df['source'] == 'crema_d']
print(f'CREMA-D test klip sayısı: {len(cre_test)}')

base = 'https://github.com/CheyneyComputerScience/CREMA-D/raw/master/VideoFlash'
failed = []
for fname in tqdm(cre_test['filename'], desc='CREMA-D video'):
    stem = fname.rsplit('.',1)[0]
    flv = f'{stem}.flv'
    local = f'{CRE_VIDEO_DIR}/{flv}'
    if os.path.exists(local) and os.path.getsize(local) > 1000:
        continue
    try:
        urllib.request.urlretrieve(f'{base}/{flv}', local)
    except Exception as e:
        failed.append((flv, str(e)[:50]))

print(f'Başarılı: {len(cre_test) - len(failed)}/{len(cre_test)}')
if failed:
    print(f'İlk 5 başarısız: {failed[:5]}')

In [ ]:
# Test setini local video path'lere eşle
def find_rav_video(audio_fname):
    # 03-01-...wav -> 02-01-...mp4
    parts = audio_fname.replace('.wav','').split('-')
    parts[0] = '02'
    video_base = '-'.join(parts) + '.mp4'
    for p in glob.glob(f'{RAV_VIDEO_DIR}/**/{video_base}', recursive=True):
        return p
    return None

def find_cre_video(audio_fname):
    flv = audio_fname.replace('.wav', '.flv')
    p = f'{CRE_VIDEO_DIR}/{flv}'
    return p if os.path.exists(p) and os.path.getsize(p) > 1000 else None

test_df['video_path'] = test_df.apply(
    lambda r: find_rav_video(r['filename']) if r['source']=='ravdess' else find_cre_video(r['filename']),
    axis=1)

found = test_df['video_path'].notna().sum()
print(f'Video eşleştirme: {found}/{len(test_df)} bulundu')

## 6. Vision predictions

In [ ]:
def predict_vision(video_path, n_frames=FRAMES_PER_CLIP):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 1:
        cap.release()
        return None
    indices = [int(total * (i+1) / (n_frames+1)) for i in range(n_frames)]
    faces = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret: continue
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        dets = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(40,40))
        if len(dets) > 0:
            x,y_,w,h = max(dets, key=lambda b: b[2]*b[3])
            face = gray[y_:y_+h, x:x+w]
        else:
            H, W_ = gray.shape
            s = min(H, W_) // 2
            face = gray[H//2-s//2:H//2+s//2, W_//2-s//2:W_//2+s//2]
        if face.size == 0: continue
        face = cv2.resize(face, (VISION_IMG_SIZE, VISION_IMG_SIZE))
        faces.append(face[..., None].astype(np.float32))
    cap.release()
    if not faces: return None
    probs = vision_model.predict(np.stack(faces), verbose=0)
    return probs.mean(axis=0)

# Sample test
valid_sample = test_df[test_df['video_path'].notna()].iloc[0]
out = predict_vision(valid_sample['video_path'])
print(f'Test örnek: {valid_sample["filename"]}, gerçek={CLASSES[valid_sample["label"]]}')
print(f'Vision probs: {out}')
print(f'Vision tahmin: {CLASSES[int(out.argmax())] if out is not None else "YOK"}')

In [ ]:
# Tüm test seti üzerinde vision
vision_probs = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Vision'):
    if pd.isna(row['video_path']):
        vision_probs.append(None)
        continue
    p = predict_vision(row['video_path'])
    vision_probs.append(p)

n_vision_ok = sum(p is not None for p in vision_probs)
print(f'\nVision OK: {n_vision_ok}/{len(test_df)}')

## 7. Fusion + metrikler

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, precision_recall_fscore_support)

# Vision'ı olan klipler için 3 koşul karşılaştır
valid = np.array([p is not None for p in vision_probs])
y_true = test_df['label'].values[valid]
vis    = np.stack([p for p in vision_probs if p is not None])
ser    = ser_probs_test[valid]
fus    = W_VISION * vis + W_SER * ser

print(f'Karşılaştırma: {valid.sum()} klipte 3 koşul\n')

results = {}
for name, preds in [('Vision-only', vis.argmax(1)),
                    ('SER-only',    ser.argmax(1)),
                    ('Fusion',      fus.argmax(1))]:
    acc = accuracy_score(y_true, preds)
    cm = confusion_matrix(y_true, preds, labels=list(range(NUM_CLASSES)))
    rep = classification_report(y_true, preds, target_names=CLASSES,
                                 labels=list(range(NUM_CLASSES)), digits=3, zero_division=0)
    results[name] = {'acc': float(acc), 'cm': cm.tolist(), 'report': rep}
    print(f'==== {name} ==== acc={acc:.4f}')
    print(rep)

# Ekstra: SER-only full test (vision'a bağlı değil)
full_ser_acc = (ser_probs_test.argmax(1) == y_test_cache).mean()
print(f'\n==== SER-only FULL test ({len(y_test_cache)} klip) ====  acc={full_ser_acc:.4f}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, (name, res) in zip(axes, results.items()):
    cm = np.array(res['cm'])
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax,
                cbar=False, vmin=0, vmax=1)
    ax.set_title(f'{name}  acc={res["acc"]:.3f}', fontsize=13)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Özet + kaydet
rows = []
for name, res in results.items():
    preds = {'Vision-only': vis.argmax(1), 'SER-only': ser.argmax(1), 'Fusion': fus.argmax(1)}[name]
    p, r, f, _ = precision_recall_fscore_support(
        y_true, preds, labels=list(range(NUM_CLASSES)), average='macro', zero_division=0)
    rows.append({'Modality': name, 'Accuracy': res['acc'],
                 'Macro-P': p, 'Macro-R': r, 'Macro-F1': f, 'N': int(valid.sum())})
rows.append({'Modality': 'SER (full test set)', 'Accuracy': full_ser_acc,
             'Macro-P': None, 'Macro-R': None, 'Macro-F1': None, 'N': len(y_test_cache)})

summary = pd.DataFrame(rows).set_index('Modality')
print(summary.to_string(float_format='%.4f'))
summary.to_csv(f'{OUT_DIR}/summary.csv')

# Per-clip detay
pred_df = test_df[valid].copy().reset_index(drop=True)
pred_df['true_label'] = pred_df['label'].map({i:c for c,i in CLASS_TO_IDX.items()})
pred_df['vision_pred'] = [CLASSES[i] for i in vis.argmax(1)]
pred_df['ser_pred']    = [CLASSES[i] for i in ser.argmax(1)]
pred_df['fusion_pred'] = [CLASSES[i] for i in fus.argmax(1)]
pred_df['vision_conf'] = vis.max(1)
pred_df['ser_conf']    = ser.max(1)
pred_df['fusion_conf'] = fus.max(1)
pred_df.to_csv(f'{OUT_DIR}/per_clip_predictions.csv', index=False)

with open(f'{OUT_DIR}/results.json', 'w') as f:
    json.dump({
        'classes': CLASSES,
        'fusion_weights': {'vision': W_VISION, 'ser': W_SER},
        'n_test_full': len(y_test_cache),
        'n_test_valid_both': int(valid.sum()),
        'ser_full_acc': float(full_ser_acc),
        'results_on_valid': {k: {'accuracy': v['acc'], 'cm': v['cm']} for k,v in results.items()},
    }, f, indent=2)

print(f'\nKaydedildi: {OUT_DIR}/')
for f in os.listdir(OUT_DIR):
    print(f'  {f}')